In [2]:
import pandas as pd
import math as m
import matplotlib.pyplot as plt
import numpy as np
import librosa
import seaborn as sns
import statistics
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report
from faster_whisper import WhisperModel 
from xgboost import XGBClassifier
import joblib
import shap
#Defining a class 
class AudioProcessor:
    #All import init method which initializing or audio,amplitude,service rate
    def __init__(self,file):
        self.file=file
        self.y=None
        self.sr=None
        
    def load_audio(self):
        #loading Audio using librosa
        self.y,self.sr=librosa.load(self.file)
        return True

    def extract_pauses(self):
        #This one will give me intervals where he/she  was talking 
        #So to get for how long he/she remained quite i need to get the intervals when he/she
        #last spoke and when he/she started speaking those are intervals of pausing
        speaking = librosa.effects.split(self.y)
        
        if(len(speaking)<2):
            return{
                'Pauses':0.0,
                'maximum pause':0.0,
                'Number of puases':0
            }
        pause_length=[]
        for i in range(1,len(speaking)):
            prev_end=speaking[i-1][1]/self.sr
            current_start=speaking[i][0]/self.sr
            pause=current_start-prev_end
            if(pause>0):
                pause_length.append(pause)
            #Meaning he/she never paused
        if(len(pause_length)==0):
            return{
                'Pauses':0.0,
                'maximum pause':0.0,
                'Number of puases':0
            }
            #rounded with float to get rid of np.float then i rounded it to 3 decimal places 
        average_pause=float(sum(pause_length)/len(pause_length))
        maximum_pause=float(max(pause_length))
        number_of_pauses=len(pause_length)
        return{
                'Pauses':round(average_pause,3),
                'maximum pause':round(maximum_pause,3),
                'Number of puases':number_of_pauses
            }
        #----------------------Exctracing Mel frequency Cepstral Co effcient-----------------------------
    def extract_mfccs(self):
        self.mfccs=librosa.feature.mfcc(y=self.y,sr=self.sr)
        self.dict={}
        for i in range(len(self.mfccs)):
            mean=round(float(statistics.mean(self.mfccs[i])),4)
            std=round(float(statistics.stdev(self.mfccs[i])),4)
            self.dict[f'mfcc_{i+1}_mean']=mean
            self.dict[f'mfcc_{i+1}_std']=std
        return self.dict
        #---------------------Extracting Chroma------------------------------------------------
    def extract_chroma(self):
        self.chroma=librosa.feature.chroma_stft(y=self.y,sr=self.sr)
        self.dict2={}
        for i in range(len(self.chroma)):
            mean=round(float(statistics.mean(self.chroma[i])),4)
            std=round(float(statistics.stdev(self.chroma[i])),4)
            self.dict2[f'chroma_{i+1}_mean']=mean
            self.dict2[f'chroma_{i+1}_std']=std
        return self.dict2
        #--------------------Extracting Spectrals-------------------------------------------
    def extract_sprectral(self):
        self.centroid=librosa.feature.spectral_centroid(y=self.y,sr=self.sr)
        self.bandwidth=librosa.feature.spectral_bandwidth(y=self.y,sr=self.sr)
        self.rolloff=librosa.feature.spectral_rolloff(y=self.y,sr=self.sr)
        self.dict3={}
        self.dict4={}
        self.dict5={}
        for i in range(len(self.centroid)):
            mean=round(float(statistics.mean(self.centroid[i])),4)
            std=round(float(statistics.stdev(self.centroid[i])),4)
            self.dict3[f'centroid__mean']=mean
            self.dict3[f'centroid__std']=std
        for i in range(len(self.bandwidth)):
            mean=round(float(statistics.mean(self.bandwidth[i])),4)
            std=round(float(statistics.stdev(self.bandwidth[i])),4)
            self.dict4[f'bandwidth__mean']=mean
            self.dict4[f'bandwidth__std']=std
        for i in range(len(self.rolloff)):
            mean=round(float(statistics.mean(self.rolloff[i])),4)
            std=round(float(statistics.stdev(self.rolloff[i])),4)
            self.dict5[f'rolloff__mean']=mean
            self.dict5[f'rolloff__std']=std
        self.dict3.update(self.dict4)
        self.dict3.update(self.dict5)
        return self.dict3
     #-------------------------Extracting Temporals(zero crodding rate and rms energy)-----------------
    def extract_temporal(self):
        zero_crossing_rate=librosa.feature.zero_crossing_rate(y=self.y)
        rms=librosa.feature.rms(y=self.y)
        self.MyDict={} 
        for i in range(len(zero_crossing_rate)):
            mean=round(float(np.mean(zero_crossing_rate[i])),4)
            std=round(float(np.std(zero_crossing_rate[i])),4)
            self.MyDict[f'Zero_crossing_rate_mean']=mean
            self.MyDict[f'Zero_crossing_rate_std']=std
        for x in range(len(rms)):
            mean=round(float(np.mean(rms)),4)
            std=round(float(np.std(rms)),4)
            self.MyDict[f'Rms_mean']=mean
            self.MyDict[f'Rms_std']=std
        return self.MyDict

    #------------------------Combining all my Methods -------------------------------
    def All_Features(self):
        self.LastDict={}
        self.LastDict.update(self.extract_pauses())
        self.LastDict.update(self.extract_mfccs())
        self.LastDict.update(self.extract_chroma())
        self.LastDict.update(self.extract_sprectral())
        self.LastDict.update(self.extract_temporal())
        return self.LastDict
        
        
#-----------------Now Building my data from recordings i created for testing--------------------------
class DataSet:
    def __init__(self):
        self.data=[]
    def add_files(self,file,label):
        self.pr=AudioProcessor(file)
        self.MyDict2={}
        self.pr.load_audio()
        self.MyDict2=self.pr.All_Features()
        self.MyDict2[f'label']=label
        self.data.append(self.MyDict2)
        return self.data
    def to_dataFrame(self):
         self.df=pd.DataFrame(self.data)
         return self.df
    def save_to_csv(self,name_of_file):
         self.df.to_csv(name_of_file,index=False)
         return name_of_file

#------------------------Data exploration -------------------------------

class DataExplore:
    def __init__(self,file):
        self.Df=pd.read_csv(file)
        self.Df=self.Df.drop(['Unnamed: 0'],axis=1)
    def check(self):
        legit=sum(self.Df['label']==0)
        fraud=sum(self.Df['label']==1)
        return legit,fraud
    def plot_all_distributions(self):
        feature_cols = self.Df.iloc[:, :-1]
        plt.title('Histogram of feature')
        for f in feature_cols:
            plt.hist(f,bins=30,color='skyblue',edgecolor='black')
            plt.show()
            #Plottting the heatmap now
        sns.heatmap(feature_cols.corr())
        plt.show()
    def scale_features(self):
        sc=StandardScaler()
        x=self.Df.drop('label',axis=1)
        y=self.Df['label']
        sc.fit(x)
        new_x=sc.transform(x)
        df2 = self.Df.copy()
        df2[x.columns]=new_x
        return df2
    def baseline_model(self):
        self.Df1 = self.Df.drop(columns=['Transcript', 'Pauses']).dropna(axis=0)
        x=self.Df1.drop(['label'],axis=1)
        y=self.Df1['label']
        X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
        model=LogisticRegression(max_iter=1000)
        model.fit(X_train,y_train)
        y_pred=model.predict(X_test)
        accuracy=round(accuracy_score(y_test,y_pred),2)
        precision=round(precision_score(y_test,y_pred),2)
        recall=round(recall_score(y_test,y_pred),2)
        f1=round(f1_score(y_test,y_pred),2)
        result={"Accuracy score :":accuracy," Precission score : ":precision," Recal score :":recall,"F1 score :":f1,"Y_pred":y_pred}
        return classification_report(y_test, y_pred)
    def dfcheck(self):
        self.Df=self.Df.drop(columns=['Pauses','maximum pause','Number of puases'])
        return self.Df.isnull().sum()*100
    def myDataframe(self):
        return self.Df.columns
    def Transcript(self): 
        trans=[]
        model = WhisperModel("base",device="cpu",compute_type="int8")
# Load audio with librosa (no FFmpeg needed)
        for i in range(1,16):
            y, sr = librosa.load(f"fraud{i}.wav", sr=16000)
            y = y.astype("float32")
            segment,info= model.transcribe(y)
            text=" ".join(t.text for t in segment)
            trans.append(text)
        for i in range(1,16):
            y, sr = librosa.load(f"legit{i}.wav", sr=16000)
            y = y.astype("float32")
            segment,info = model.transcribe(y)
            text=" ".join(k.text for k in segment)
            trans.append(text)
        self.Df['Transcript']=trans
        return self.Df
    def extract_text_features(self):
#Extract hesitation scenirios when he thinks of lies or not sure about the answer
        not_sure1=[]
        num_of_words1=[]
        maybe1=[]
        forgot1=[]
        i_think1=[]
        sentence_length1=[]
        for x in self.Df['Transcript']:
            number_of_words=len(x.split())
            not_sure=x.lower().count('not sure')
            maybe=x.lower().count('maybe')
            forgot=x.lower().count('forgot')
            i_think=x.lower().count('i think')
            sentence_length=len(x.split()) / max(x.count('.'), 1)
            not_sure1.append(not_sure)
            num_of_words1.append(number_of_words)
            maybe1.append(maybe)
            forgot1.append(forgot)
            i_think1.append(i_think)
            sentence_length1.append(sentence_length)
        self.Df['not_sure']=not_sure1
        self.Df['num_of_words']=num_of_words1
        self.Df['maybe']=maybe1
        self.Df['forgot']=forgot1
        self.Df['i_think']=i_think1
        self.Df['sentence_length']=sentence_length1
        return self.Df
    def Explonetory_data_for_text_features(self):
        self.temp_mean1=self.Df[self.Df['label']==1][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']].mean()
        self.temp_mean2=self.Df[self.Df['label']==0][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']].mean()
        self.temp_var1=self.Df[self.Df['label']==1][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']].std()
        self.temp_var2=self.Df[self.Df['label']==0][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']].std()
        print("Fraude Data exploration","\n","Mean :",self.temp_mean1,"\n","Varience :",self.temp_var1,'\n')
        print("Legit Data exploration","\n","Mean :",self.temp_mean2,"\n","Varience :",self.temp_var2,'\n')

        if(self.temp_mean1>self.temp_mean2).all():
            print("Frauds have hesitated more than the legits calls \n")
        elif(self.temp_mean1==self.temp_mean2).all():
            print("Frauds hasitate same amount of time as legits \n")
        else:
            print("Legits have hasitated more  amount of time than frauds \n")
        x=self.Df[self.Df['label']==1][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']]
        x.sum().plot(kind='bar')
        plt.title("Fraude Bar plots")
        plt.xlabel('Hesitant words')
        plt.ylabel("Count")
        plt.show()
        x1=self.Df[self.Df['label']==0][['not_sure','num_of_words','maybe','forgot','i_think','sentence_length']]
        x1.sum().plot(kind='bar')
        plt.title("Legit Bar plots")
        plt.xlabel('Hesitant words')
        plt.ylabel("Count")
        plt.show()
        
        #Box plots 
        fig, ax = plt.subplots(figsize=(8, 6))
        data= [x['not_sure'], x['num_of_words'], x['maybe'], x['forgot'], x['i_think'], x['sentence_length']]
        ax.boxplot(data)
        ax.set_title("Box plot for frauds")
        ax.set_xlabel("Columns")
        ax.set_ylabel("values")
        ax.set_xticklabels(['not_sure','num_of_words','maybe','forgot','i_think','sentence_length'])
        plt.show()
        
        fig, ax = plt.subplots(figsize=(8, 6))
        data2 = [x1['not_sure'], x1['num_of_words'], x1['maybe'], x1['forgot'], x1['i_think'], x1['sentence_length']]
        ax.boxplot(data2)
        ax.set_title("Box plot for legit")
        ax.set_xlabel("Columns")
        ax.set_ylabel("values")
        ax.set_xticklabels(['not_sure','num_of_words','maybe','forgot','i_think','sentence_length'])
        plt.show()
    def xgbmodel(self):
        self.Df1 = self.Df.drop(columns=['Transcript', 'Pauses']).dropna(axis=0)  
        self.x=self.Df1.drop(['label'],axis=1)
        self.y=self.Df1['label']
        x_train,x_test,y_train,y_test=train_test_split(self.x,self.y,test_size=0.2,random_state=42)
        self.model=XGBClassifier(n_estimators=200,max_depth=7,learning_rate=0.05)
        self.model.fit(x_train,y_train)

        #saving the model 
        joblib.dump(self.model,'xgbooster_fraud_model.pkl')
        y_pred=self.model.predict(x_test)
        report= classification_report(y_test,y_pred,output_dict=True)  #output_dict=True will return Dictionary instead of string
        df_report=pd.DataFrame(report).transpose()
        df_report.to_csv("Classi report n_esti=200")
        return classification_report(y_test,y_pred)
    def important_model_features(self):
        importances=self.model.feature_importances_
        print(importances.shape,"    ",len(self.x.columns))
        importances = np.array(importances).ravel()
        small_dataframe=pd.Series(data=importances,index=self.x.columns)
        small_dataframe = small_dataframe.sort_values(ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(16, 6))
        ax.bar(range(len(small_dataframe)), small_dataframe.values)
        ax.set_xticks(range(len(small_dataframe)))
        ax.set_xticklabels(small_dataframe.index, rotation=90, fontsize=8)
        ax.set_title("Top 20 Feature Importances - XGBoost")
        ax.set_xlabel("Features")
        ax.set_ylabel("Importance Score")
        plt.tight_layout()
        plt.show()
    def shap_explaining(self,index):
        loaded_model=joblib.load("xgbooster_fraud_model.pkl")
        explainer=shap.TreeExplainer(loaded_model)
        row=self.x.iloc[[index]]
        shap_values = explainer.shap_values(row)
        shap.initjs()
        shap.force_plot(base_value=explainer.expected_value, shap_values=shap_values[0], features=row)
        return shap_values
        
        
        
